In [7]:
from numba import njit
import numpy as np

In [8]:
@njit
def get_indices_3_ifo_2(tlen, t2_coinc_window, t3_coinc_window, num_combinations, dtype=np.int64):
    idx = np.empty((num_combinations, 3), dtype=dtype)
    row_num = 0
    for i in range(tlen):
        for j in range(
            max(i - t2_coinc_window, 0), min(tlen, i + t2_coinc_window + 1)
        ):
            
            for k in range(
                max(i - t3_coinc_window, 0), min(tlen, i + t3_coinc_window + 1)
            ):
                idx[row_num] = [i, j, k]
                row_num += 1
    return idx

def number_tail(t2_coinc_window, t3_coinc_window):
    n_tail = sum(
        (t2_coinc_window + 1 + np.arange(0, t2_coinc_window + 1))
        * (t3_coinc_window + 1 + np.arange(0, t2_coinc_window + 1))
    ) + sum(
        (2 * t2_coinc_window + 1)
        * (t3_coinc_window + 1 + np.arange(t2_coinc_window + 1, t3_coinc_window + 1))
    )
    return n_tail

def number_coincident_combinations(tlen, t2_coinc_window, t3_coinc_window):
    """Assumes t3_coinc_window>t2_coinc_window, and tlen is the number
    of time samples.
    """
    n_tail = number_tail(t2_coinc_window, t3_coinc_window)
    T_2 = 2 * t2_coinc_window + 1
    T_3 = 2 * t3_coinc_window + 1
    n_middle = (tlen - 2 * (t3_coinc_window + 1)) * T_2 * T_3
    n_c = 2 * n_tail + n_middle
    return n_c

def number_coincident_combinations_parts(tlen, t2_coinc_window, t3_coinc_window):
    """Assumes t3_coinc_window>t2_coinc_window, and tlen is the number
    of time samples.
    """
    n_tail = number_tail(t2_coinc_window, t3_coinc_window)
    T_2 = 2 * t2_coinc_window + 1
    T_3 = 2 * t3_coinc_window + 1
    n_middle = (tlen - 2 * (t3_coinc_window + 1)) * T_2 * T_3
    return n_tail, n_middle

# @njit
# def get_indices_3_ifo_3(tlen, t2_coinc_window, t3_coinc_window, n_tail, n_middle, dtype=np.int64):
#     idx = np.empty((num_combinations, 3), dtype=dtype)
#     row_num = 0
#     n_tail = 
#     for i in range(tlen):
#         for j in range(
#             max(i - t2_coinc_window, 0), min(tlen, i + t2_coinc_window + 1)
#         ):
            
#             for k in range(
#                 max(i - t3_coinc_window, 0), min(tlen, i + t3_coinc_window + 1)
#             ):
#                 idx[row_num] = [i, j, k]
#                 row_num += 1
#     return idx

In [20]:
import benchmark2
from importlib import reload
reload(benchmark2)

<module 'benchmark2' from '/Users/camill/projects/pycbc/test_hm/playground/multi_inspiral_dev/indices_speedup/benchmark2.py'>

In [29]:
tlen = 20
t2_coinc_window = 1
t3_coinc_window = 2
num_combinations = number_coincident_combinations(tlen, t2_coinc_window, t3_coinc_window)
# FIXME: n_tail should be 18, not 33. 
n_tail, n_middle = number_coincident_combinations_parts(tlen, t2_coinc_window, t3_coinc_window)
idx = benchmark2.get_indices_3_ifo_3(tlen, t2_coinc_window, t3_coinc_window, n_tail, n_middle)
idx_correct = benchmark2.get_indices_3_ifo_2(tlen, t2_coinc_window, t3_coinc_window, num_combinations)
# idx_correct = idx.copy()
(idx_correct[n_tail:-n_tail] == idx[n_tail:-n_tail]).all()

True

In [26]:
idx_correct[18:-18]

array([[ 2,  1,  0],
       [ 2,  1,  1],
       [ 2,  1,  2],
       [ 2,  1,  3],
       [ 2,  1,  4],
       [ 2,  2,  0],
       [ 2,  2,  1],
       [ 2,  2,  2],
       [ 2,  2,  3],
       [ 2,  2,  4],
       [ 2,  3,  0],
       [ 2,  3,  1],
       [ 2,  3,  2],
       [ 2,  3,  3],
       [ 2,  3,  4],
       [ 3,  2,  1],
       [ 3,  2,  2],
       [ 3,  2,  3],
       [ 3,  2,  4],
       [ 3,  2,  5],
       [ 3,  3,  1],
       [ 3,  3,  2],
       [ 3,  3,  3],
       [ 3,  3,  4],
       [ 3,  3,  5],
       [ 3,  4,  1],
       [ 3,  4,  2],
       [ 3,  4,  3],
       [ 3,  4,  4],
       [ 3,  4,  5],
       [ 4,  3,  2],
       [ 4,  3,  3],
       [ 4,  3,  4],
       [ 4,  3,  5],
       [ 4,  3,  6],
       [ 4,  4,  2],
       [ 4,  4,  3],
       [ 4,  4,  4],
       [ 4,  4,  5],
       [ 4,  4,  6],
       [ 4,  5,  2],
       [ 4,  5,  3],
       [ 4,  5,  4],
       [ 4,  5,  5],
       [ 4,  5,  6],
       [ 5,  4,  3],
       [ 5,  4,  4],
       [ 5,  

In [14]:
for j in range( -t2_coinc_window, t2_coinc_window + 1):
    for k in range(-t3_coinc_window, t3_coinc_window + 1):
        print(j, k)

-1 -2
-1 -1
-1 0
-1 1
-1 2
0 -2
0 -1
0 0
0 1
0 2
1 -2
1 -1
1 0
1 1
1 2


In [50]:
itertools.product??

In [51]:
range(-t2_coinc_window, t2_coinc_window + 1)

range(-1, 2)

In [150]:
@njit
def product(a, b):
    return np.array([[i, j] for i in a for j in b])

In [121]:
@njit
def product(a, b):
    idx = np.empty((len(a) * len(b), 2), dtype=np.int64)
    row_num = 0
    for i in a:
        for j in b:
            idx[row_num] = [i, j]
            row_num += 1
    return idx

In [161]:
# @njit
def get_indices_3_ifo_3(tlen, t3_coinc_window, i_j_vals, n_tail, n_middle, dtype=np.int64):
    num_combinations = 2 * n_tail + n_middle
    idx = np.empty((num_combinations, 3), dtype=dtype)
    
    row_num = n_tail
    while row_num < num_combinations - n_tail:
        for i in prange(t3_coinc_window + 1, tlen - t3_coinc_window - 1):
            # for j in prange(-t2_coinc_window, t2_coinc_window+1):
            #     for k in prange(-t3_coinc_window, t3_coinc_window+1):
            for j, k in i_j_vals:
                idx[row_num] = [i, j+i, k+i]
                row_num += 1
    return idx

In [ ]:
np.arange(-args.t2_coinc_window, args.t2_coinc_window+1), np.arange(-args.t2_coinc_window, args.t2_coinc_window+1)

In [ ]:
i_j_vals = product()

In [ ]:
get_indices_3_ifo_3(5000, 17, i_j_vals, 7377 2953580)

In [159]:
# time product
t0 = time.time()
product(np.arange(9),
        np.arange(9))
t1 = time.time()
print((t1 - t0)*1000) 

0.5717277526855469


In [56]:
import time

0.15223002433776855


In [54]:
for j, k in product(np.arange(-t2_coinc_window, t2_coinc_window + 1),
                              np.arange(-t3_coinc_window, t3_coinc_window + 1)):
    print(j, k)

-1 -2
-1 -1
-1 0
-1 1
-1 2
0 -2
0 -1
0 0
0 1
0 2
1 -2
1 -1
1 0
1 1
1 2


In [30]:
for j, k in itertools.product(range(-t2_coinc_window, t2_coinc_window + 1),
                              range(-t3_coinc_window, t3_coinc_window + 1)):
    print(j, k)

-1 -2
-1 -1
-1 0
-1 1
-1 2
0 -2
0 -1
0 0
0 1
0 2
1 -2
1 -1
1 0
1 1
1 2


In [49]:
m = np.meshgrid(range(-t2_coinc_window, t2_coinc_window + 1),
                              range(-t3_coinc_window, t3_coinc_window + 1))
m[0].T.ravel(), m[1].T.ravel()

(array([-1, -1, -1, -1, -1,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1]),
 array([-2, -1,  0,  1,  2, -2, -1,  0,  1,  2, -2, -1,  0,  1,  2]))

In [37]:
for i in zip(np.meshgrid(range(-t2_coinc_window, t2_coinc_window + 1),
                              range(-t3_coinc_window, t3_coinc_window + 1))):
    print(i)

(array([[-1,  0,  1],
       [-1,  0,  1],
       [-1,  0,  1],
       [-1,  0,  1],
       [-1,  0,  1]]),)
(array([[-2, -2, -2],
       [-1, -1, -1],
       [ 0,  0,  0],
       [ 1,  1,  1],
       [ 2,  2,  2]]),)


In [36]:
for j, k in zip(np.meshgrid(range(-t2_coinc_window, t2_coinc_window + 1),
                              range(-t3_coinc_window, t3_coinc_window + 1))):
    print(j, k)

ValueError: not enough values to unpack (expected 2, got 1)

In [16]:
import itertools
from itertools import product

In [ ]:
product()

In [34]:
idx_correct[:,0]

array([ 0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  3,
        3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  4,  4,  4,
        4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  5,  5,  5,  5,  5,
        5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  6,  6,  6,  6,  6,  6,  6,
        6,  6,  6,  6,  6,  6,  6,  6,  7,  7,  7,  7,  7,  7,  7,  7,  7,
        7,  7,  7,  7,  7,  7,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,
        8,  8,  8,  8,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,
        9,  9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12,
       12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 13, 13, 13, 13,
       13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14, 14,
       14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 15, 15, 15,
       15, 15, 15, 15, 15

In [38]:
18

18

In [40]:
u = np.unique(idx_correct[:,0][18:-18], return_counts=True)
u

(array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17]),
 array([15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15]))